## Imports

In [1]:
import pandas as pd
import json

## Helper functions

In [2]:
def load_json(path):
  with open(path, 'r', encoding='utf-8') as f:
    obj = json.load(f)
    return obj

def inspect_json(obj, label):
  print(f'{label} loaded type: {type(obj)}')
  if isinstance(obj, dict):
    print("Top level keys: ", list(obj.keys()))

def extract_records(obj, key_name):
  if isinstance(obj, list):
    return obj
  elif isinstance(obj, dict):
    return obj[key_name]
  else:
    raise ValueError("Unexpected JSON structure.")

def preview_df(df, label):
  print(f'{label}:')
  print("Rows: ", len(df))
  print("Columns: ", df.columns.tolist())
  print("Shape: ", df.shape)

  if not df.empty:
    print("First row: ")
    print(df.iloc[0].to_dict())
  else:
    print("DataFrame is empty.")


def save_csv(df, out_path):
  df.to_csv(out_path, index=False)

## Load raw JSON

In [3]:
carts_obj = load_json("../raw_data/carts.json")
products_obj = load_json("../raw_data/products.json")
users_obj = load_json("../raw_data/users.json")

##Inspect and extract records

In [4]:
inspect_json(carts_obj, "Carts")
inspect_json(products_obj, "Products")
inspect_json(users_obj, "Users")


Carts loaded type: <class 'dict'>
Top level keys:  ['carts', 'total', 'skip', 'limit']
Products loaded type: <class 'dict'>
Top level keys:  ['products', 'total', 'skip', 'limit']
Users loaded type: <class 'dict'>
Top level keys:  ['users', 'total', 'skip', 'limit']


In [5]:
carts_records = extract_records(carts_obj, 'carts')
products_records = extract_records(products_obj, 'products')
users_records = extract_records(users_obj, 'users')

## Build DataFrame

In [6]:
carts_df = pd.DataFrame(carts_records)
products_df = pd.DataFrame(products_records)
users_df = pd.DataFrame(users_records)

In [7]:
preview_df(carts_df, "Carts")
preview_df(products_df, "Products")
preview_df(users_df, "Users")

Carts:
Rows:  50
Columns:  ['id', 'products', 'total', 'discountedTotal', 'userId', 'totalProducts', 'totalQuantity']
Shape:  (50, 7)
First row: 
{'id': 1, 'products': [{'id': 168, 'title': 'Charger SXT RWD', 'price': 32999.99, 'quantity': 3, 'total': 98999.97, 'discountPercentage': 13.39, 'discountedTotal': 85743.87, 'thumbnail': 'https://cdn.dummyjson.com/products/images/vehicle/Charger%20SXT%20RWD/thumbnail.png'}, {'id': 78, 'title': 'Apple MacBook Pro 14 Inch Space Grey', 'price': 1999.99, 'quantity': 2, 'total': 3999.98, 'discountPercentage': 18.52, 'discountedTotal': 3259.18, 'thumbnail': 'https://cdn.dummyjson.com/products/images/laptops/Apple%20MacBook%20Pro%2014%20Inch%20Space%20Grey/thumbnail.png'}, {'id': 183, 'title': 'Green Oval Earring', 'price': 24.99, 'quantity': 5, 'total': 124.94999999999999, 'discountPercentage': 6.28, 'discountedTotal': 117.1, 'thumbnail': 'https://cdn.dummyjson.com/products/images/womens-jewellery/Green%20Oval%20Earring/thumbnail.png'}, {'id': 100,

## Clean and transform

In [8]:
# Remove accidental leading/trailing spaces from raw column names
carts_df.columns = carts_df.columns.str.strip()
products_df.columns = products_df.columns.str.strip()
users_df.columns = users_df.columns.str.strip()

In [9]:
#Keep only columns needed for validation and aggregation
products_df = products_df[['id', 'category', 'rating', 'stock', 'tags', 'brand']]
users_df = users_df[['id', 'firstName', 'lastName', 'age', 'gender', 'username']]

# Rename columns for clarity and consistency (snake_case)
carts_df = carts_df.rename(columns={
    'id': 'cart_id',
    'userId': 'user_id',
    'total': 'cart_total',
    'totalProducts': 'cart_total_products',
    'totalQuantity': 'cart_total_quantity',
    'discountedTotal': 'cart_discounted_total'
})

products_df = products_df.rename(columns={
    'id': 'product_id',
    'category': 'product_category',
    'rating': 'product_rating',
    'stock': 'product_stock',
    'tags': 'product_tags',
    'brand': 'product_brand'
})

users_df = users_df.rename(columns={
    'id': 'user_id',
    'firstName': 'user_first_name',
    'lastName': 'user_last_name',
    'age': 'user_age',
    'gender': 'user_gender'
})


In [10]:
## Explode nested cart products so each row represents one cart item

carts_df = carts_df.explode('products').reset_index(drop=True)

In [11]:
#Flatten item dictionaries into columns
product_cols = pd.json_normalize(carts_df['products'])
carts_df = pd.concat([carts_df.drop(columns=['products']), product_cols], axis=1)

carts_df = carts_df.rename(columns={
    'id': 'product_id',
    'title': 'product_title',
    'price': 'product_price',
    'quantity': 'product_quantity',
    'total': 'product_total',
    'discountPercentage': 'product_discount_percentage',
    'discountedTotal': 'product_discounted_total'
})

carts_df = carts_df.drop(columns=['thumbnail'], errors='ignore')

In [12]:

carts_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   cart_id                      198 non-null    int64  
 1   cart_total                   198 non-null    float64
 2   cart_discounted_total        198 non-null    float64
 3   user_id                      198 non-null    int64  
 4   cart_total_products          198 non-null    int64  
 5   cart_total_quantity          198 non-null    int64  
 6   product_id                   198 non-null    int64  
 7   product_title                198 non-null    object 
 8   product_price                198 non-null    float64
 9   product_quantity             198 non-null    int64  
 10  product_total                198 non-null    float64
 11  product_discount_percentage  198 non-null    float64
 12  product_discounted_total     198 non-null    float64
dtypes: float64(6), int64

## Validate

In [13]:
carts_bad_mask = (
    carts_df['cart_id'].isna() |
    carts_df['user_id'].isna() |
    carts_df['product_id'].isna() |
    carts_df['product_quantity'].isna() |
    (carts_df['product_quantity'] <= 0)
)

In [14]:
clean_carts = carts_df[~carts_bad_mask]
bad_carts = carts_df[carts_bad_mask]
bad_carts['source_table'] = 'carts'
bad_carts['reject_reason'] = 'missing or invalid cart/product data'

#Enrich carts data with product and user data
merged_df = clean_carts.merge(products_df, on='product_id', how='left')
merged_df = merged_df.merge(users_df, on='user_id', how='left')

merged_df['product_brand'] = merged_df['product_brand'].fillna('Unknown')

In [15]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   cart_id                      198 non-null    int64  
 1   cart_total                   198 non-null    float64
 2   cart_discounted_total        198 non-null    float64
 3   user_id                      198 non-null    int64  
 4   cart_total_products          198 non-null    int64  
 5   cart_total_quantity          198 non-null    int64  
 6   product_id                   198 non-null    int64  
 7   product_title                198 non-null    object 
 8   product_price                198 non-null    float64
 9   product_quantity             198 non-null    int64  
 10  product_total                198 non-null    float64
 11  product_discount_percentage  198 non-null    float64
 12  product_discounted_total     198 non-null    float64
 13  product_category    

## Aggregate

In [16]:
user_summary = (
    merged_df.groupby('user_id')
    .agg(
        total_quantity=('product_quantity', 'sum'),
        total_value_spent=('product_total', 'sum'),
        unique_carts=('cart_id', 'nunique'),
        average_product_price=('product_price', 'mean')
    )
    .reset_index()
)

user_lookup = merged_df[['user_id', 'username']].drop_duplicates()

user_summary = user_summary.merge(user_lookup, on='user_id', how='left')
user_summary = user_summary[[
    'user_id',
    'username',
    'total_quantity',
    'total_value_spent',
    'unique_carts',
    'average_product_price'
]]

category_summary = (
    merged_df.groupby('product_category')
    .agg(
        total_quantity_sold=('product_quantity', 'sum'),
        total_revenue=('product_total', 'sum'),
        avg_product_rating=('product_rating', 'mean'),
        unique_products=('product_id', 'nunique')
    )
    .reset_index()
)


# Optional exploratory summaries kept in the notebook but not exported in the final portfolio version
cart_summary = (
    merged_df.groupby('cart_id')
    .agg(
        total_cart_quantity=('cart_total_quantity', 'first'),
        total_cart_value=('cart_total', 'first'),
        total_cart_discounted_value=('cart_discounted_total', 'first'),
        distinct_cart_products=('product_id', 'nunique')
    )
    .reset_index()
)

sanity_check = pd.DataFrame({
    'total_revenue': [merged_df['product_total'].sum()],
    'total_quantity': [merged_df['product_quantity'].sum()],
    'average_price': [merged_df['product_price'].mean()]
})

brand_summary = (
    merged_df.groupby('product_brand')
    .agg(
        total_revenue=('product_total', 'sum'),
        items_sold=('product_quantity', 'sum'),
        avg_rating=('product_rating', 'mean')
    )
    .reset_index()
)

cart_level_df = merged_df[
    ['cart_id', 'user_gender', 'cart_total', 'cart_discounted_total', 'cart_total_quantity']
].drop_duplicates()

gender_summary = (
    cart_level_df.groupby('user_gender')
    .agg(
        total_spend=('cart_total', 'sum'),
        total_discounted_spend=('cart_discounted_total', 'sum'),
        avg_spend_per_cart=('cart_total', 'mean'),
        total_quantity_per_user=('cart_total_quantity', 'sum')
    )
    .reset_index()
)

discount_summary = (
    merged_df.groupby('product_category')
    .agg(
        avg_discount_percentage=('product_discount_percentage', 'mean'),
        total_discounted_value=('product_discounted_total', 'sum'),
        total_original_value=('product_total', 'sum')
    )
    .reset_index()
)

## Save CSV

In [17]:
save_csv(merged_df, 'clean_cart_items.csv')
save_csv(bad_carts, 'bad_records.csv')
save_csv(user_summary, 'user_summary.csv')
save_csv(category_summary, 'category_summary.csv')

In [18]:
preview_df(merged_df, "Clean cart items")
preview_df(bad_carts, "Bad records")
preview_df(user_summary, "User summary")
preview_df(category_summary, "Category summary")

Clean cart items:
Rows:  198
Columns:  ['cart_id', 'cart_total', 'cart_discounted_total', 'user_id', 'cart_total_products', 'cart_total_quantity', 'product_id', 'product_title', 'product_price', 'product_quantity', 'product_total', 'product_discount_percentage', 'product_discounted_total', 'product_category', 'product_rating', 'product_stock', 'product_tags', 'product_brand', 'user_first_name', 'user_last_name', 'user_age', 'user_gender', 'username']
Shape:  (198, 23)
First row: 
{'cart_id': 1, 'cart_total': 103774.85, 'cart_discounted_total': 89686.65, 'user_id': 33, 'cart_total_products': 4, 'cart_total_quantity': 15, 'product_id': 168, 'product_title': 'Charger SXT RWD', 'product_price': 32999.99, 'product_quantity': 3, 'product_total': 98999.97, 'product_discount_percentage': 13.39, 'product_discounted_total': 85743.87, 'product_category': 'vehicle', 'product_rating': 2.58, 'product_stock': 57, 'product_tags': ['sedans', 'sports cars', 'vehicles'], 'product_brand': 'Dodge', 'user_f

---

## 📊 Data Processing Summary

This notebook processes cart, product, and user JSON data sourced from the DummyJSON API to create a clean, analysis-ready cart item dataset and a small set of summary outputs.

### Data Loading

- Loaded raw JSON data for carts, products, and users.
- Used helper functions to inspect JSON structure and extract records into pandas DataFrames.

### Transformation

- Flattened nested cart data by exploding the `products` field so each row represents one cart item.
- Normalized product-level fields into separate columns.
- Renamed columns using consistent `snake_case` naming for clarity.
- Kept only the fields needed for validation, enrichment, and summary analysis.

### Data Integration

- Merged cart item data with product data on `product_id`.
- Merged the enriched cart data with user data on `user_id`.
- Produced a final dataset where each row represents one cart item enriched with product and user attributes.

### Validation

- Flagged invalid records based on:
  - missing `cart_id`, `user_id`, or `product_id`
  - missing or non-positive `product_quantity`
- Separated invalid rows into a dedicated `bad_records` output with a `reject_reason`.

### Aggregations

Created summary tables to support analysis:

- **User Summary**: total quantity, total spend, number of carts, and average product price per user
- **Category Summary**: total quantity sold, total revenue, average product rating, and unique products per category

Additional summaries were explored in the notebook, but the final portfolio version focuses on the outputs most relevant to validation and analysis.

### Outputs

The following datasets were saved in the final version:

- `clean_cart_items.csv` → cleaned, enriched cart item-level dataset
- `bad_records.csv` → invalid records with rejection reasons
- `user_summary.csv` → user-level summary table
- `category_summary.csv` → category-level summary table

---

## 🧠 Assumptions

- Each row in the final dataset represents one cart item.
- Cart-level fields such as `cart_total` are repeated across item rows and should be handled carefully in aggregations.
- Product totals are used as provided in the source data.
- Missing `product_brand` values are filled with `"Unknown"` in the enriched dataset.
- Validation in this version is limited to basic presence and positivity checks.

---

## 🔍 Notes

- This project focuses on data cleaning, validation, enrichment, and summary generation rather than full production pipeline design.
- The dataset is treated as internally consistent for this analysis.
- Additional checks, such as duplicate handling or cross-checking cart totals against item totals, could be added in a future version.

---